Plik tworzy aplikację Gradio wykorzystującą CrewAI do wyszukiwania i oceny hoteli, z dwoma agentami: ekspertem ds. hoteli (wyszukuje opcje) i analitykiem jakości (ocenia i wybiera najlepsze). System przyjmuje parametry podróży ,  używa narzędzi do przeszukiwania internetu i zwraca rekomendacje.

# Setup

In [ ]:
!pip install -qqU langchain_community crewai crewai_tools gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 6.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 79.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 308.7/308.7 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 89.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 583.5/583.5 kB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.1/54.1 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.9/322.9 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.3/135.3 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 8.9 MB/s et

*   `langchain_community` jest częścią frameworku LangChain, służącego do tworzenia aplikacji wykorzystujących modele językowe.
*   `crewai` to biblioteka umożliwiająca budowanie systemów opartych na współpracy wielu agentów AI.
*   `crewai_tools` zawiera narzędzia rozszerzające możliwości `crewai`.
*   `gradio` służy do tworzenia interfejsów użytkownika dla modeli uczenia maszynowego i aplikacji AI.

In [ ]:
# Standard library imports
import os
import warnings

# Third-party imports
from crewai import Agent, Task, Crew
from crewai.process import Process
from crewai_tools import SerperDevTool, ScrapeWebsiteTool, WebsiteSearchTool
from langchain_openai import ChatOpenAI

from IPython.display import Markdown

from google.colab import userdata

# Suppress warnings
warnings.filterwarnings('ignore')

import gradio as gr

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:623: UserWarning: <built-in function callable> is not a Python type (it may be an instance of an object), Pydantic will allow any object with no validation since we cannot even enforce that the input is an instance of the given type. To get rid of this error wrap the type with `pydantic.SkipValidation`.
  warn(


Ten kod zawiera listę importowanych bibliotek i modułów Pythona, które będą używane w programie. Można je podzielić na kilka kategorii:

**Biblioteki standardowe:**

*   `os`: Zapewnia funkcje do interakcji z systemem operacyjnym, takie jak dostęp do zmiennych środowiskowych i zarządzanie plikami.
*   `warnings`: Pozwala na kontrolowanie wyświetlania ostrzeżeń podczas działania programu.

**Biblioteki zewnętrzne (instalowane przez pip):**

*   `crewai`: Biblioteka do tworzenia systemów opartych na współpracy agentów AI. Zawiera klasy takie jak `Agent`, `Task` i `Crew`.
*   `crewai.process`: Moduł z funkcjami zarządzającymi procesem wykonywania zadań przez agentów w `crewai`.
*   `crewai_tools`: Zestaw narzędzi rozszerzających możliwości agentów `crewai`, takich jak:
    *   `SerperDevTool`: Narzędzie do wyszukiwania informacji w internecie za pomocą Serper API.
    *   `ScrapeWebsiteTool`: Narzędzie do pobierania zawartości stron internetowych.
    *   `WebsiteSearchTool`: Narzędzie do przeszukiwania konkretnych stron internetowych.
*   `IPython.display`: Moduł umożliwiający wyświetlanie bogatych treści w środowisku IPython (np. Jupyter Notebook), w tym formatowania Markdown.
*   `google.colab.userdata`: Pozwala na dostęp do danych użytkownika przechowywanych w Google Colab.
*   `gradio`: Biblioteka do tworzenia interfejsów graficznych dla modeli uczenia maszynowego i aplikacji AI.

Dodatkowo, kod zawiera instrukcję wyciszenia ostrzeżeń (`warnings.filterwarnings('ignore')`), co oznacza, że program nie będzie wyświetlał żadnych ostrzeżeń podczas działania.

In [ ]:
class CFG:
    model = 'gpt-4o-mini'
    temp = 0

In [ ]:
os.environ["OPENAI_API_KEY"] = userdata.get('openaivision')
os.environ["SERPER_API_KEY"] = userdata.get('serper')

# Załoga

In [ ]:
def run_travel_agent_crew(location, check_in, check_out, number_of_people, budget):

    # llm
    llm = ChatOpenAI(model= CFG.model, temperature = CFG.temp)

    # tools
    serper_search_tool = SerperDevTool()
    scrap_tool = ScrapeWebsiteTool()
    website_search_tool = WebsiteSearchTool()
    tools=[serper_search_tool, scrap_tool, website_search_tool]

    # agent definitions
    travel_expert = Agent( role='Specjalista ds. Hoteli',
      goal='Sprawnie wyszukiwać i tworzyć kompleksową listę odpowiednich opcji hotelowych w określonej lokalizacji  \
      przestrzegając podanych kryteriów, takich jak daty zameldowania/wymeldowania, liczba gości i wszelkie inne szczególne wymagania. \
      Zapewnić, że zebrane informacje są dokładne, aktualne i zawierają kluczowe szczegóły, jak ceny, udogodnienia i oceny gości.',
      backstory='Jesteś doświadczonym cyfrowym concierge podróży z wyczuleniem na szczegóły i pasją do znajdowania idealnego zakwaterowania. \
      Dzięki wieloletniemu doświadczeniu w branży hotelarskiej i rozległej wiedzy o światowych sieciach hoteli oraz butikowych obiektach, \
      doskonalisz swoje umiejętności w poruszaniu się po różnych platformach rezerwacyjnych i bazach danych hoteli. \
      Twoja specjalizacja polega na szybkim przeglądaniu licznych opcji w celu identyfikacji najbardziej odpowiednich wyborów dla podróżnych, \
      biorąc pod uwagę czynniki takie jak lokalizacja, cena, udogodnienia i opinie gości.',
      verbose=True, allow_delegation=False, llm=llm, tools=tools)

    hotel_reviewer = Agent(
      role='Analityk Jakości Hoteli',
      goal='Dokładnie oceniać i porównywać opcje hotelowe, aby dostarczać szczegółowych, bezstronnych ocen pomagających podróżnym w podejmowaniu świadomych decyzji. \
      Analizować udogodnienia, lokalizację, stosunek jakości do ceny i doświadczenia gości każdego hotelu, aby tworzyć kompleksowe recenzje podkreślające mocne strony, \
      słabości i unikalne cechy.',
      backstory = "Jesteś doświadczonym profesjonalistą branży turystycznej z ponad dziesięcioletnim doświadczeniem w ocenie i krytyce hoteli. \
      Twoje doświadczenie obejmuje pracę jako inspektor hoteli luksusowych, dziennikarz turystyczny dla znanych publikacji oraz konsultant systemów oceny hoteli.  \
      To różnorodne doświadczenie rozwinęło Twoją zdolność do oceny zakwaterowania z wielu perspektyw, uwzględniając zarówno obiektywne kryteria, \
      jak i subiektywne doświadczenia gości.   Masz szczególny talent do identyfikowania ukrytych perełek i dostrzegania potencjalnych problemów, które mogą wpłynąć na pobyt gościa.",
      verbose=True, allow_delegation=False, llm=llm, tools=tools)

    # tasks
    hotel_search_task = Task(
            description = "Wyszukaj 5 hoteli w {location} dla {number_of_people} osób dorosłych, z zameldowaniem {check_in} i wymeldowaniem {check_out}.",
            agent = travel_expert, expected_output = "Wszystkie szczegóły dotyczące konkretnie wybranego zakwaterowania."
      )

    hotel_review_task = Task(
        description="Na podstawie dostarczonych rekomendacji, wybierz najlepsze opcje w oparciu o oceny, recenzje i dostępne udogodnienia. \
        Weź pod uwagę, że budżet wynosi {budget} PLN. Postaraj się znaleźć zakwaterowanie w granicach tego budżetu.",
        expected_output="Wszystkie szczegóły dotyczące konkretnie wybranego zakwaterowania, w tym cena, URL i ewentualne zdjęcia, jeśli są dostępne.",
        agent = hotel_reviewer)


    # Create the Crew
    travel_agent_crew = Crew( agents=[travel_expert, hotel_reviewer],
    tasks=[hotel_search_task, hotel_review_task],
    verbose=True, process=Process.sequential)
    event_criteria = {
        'location': location,
        'check_in': check_in,
        'check_out': check_out,
        'number_of_people': int(number_of_people),
        'budget': float(budget)
    }

    result = travel_agent_crew.kickoff(inputs=event_criteria)

    # Convert the result to Markdown
    markdown_result = Markdown(result.raw).data
    return markdown_result


In [ ]:
# Create Gradio interface
iface = gr.Interface(
    fn=run_travel_agent_crew,
    inputs=[
        gr.Textbox(label="Location"),
        gr.Textbox(label="Check-in Date"),
        gr.Textbox(label="Check-out Date"),
        gr.Number(label="Number of People"),
        gr.Number(label="Budget")
    ],
    outputs=gr.Markdown(label="Result"),
    title="Travel Agent Crew",
    description="Enter your travel details to get hotel recommendations.",
    allow_flagging="never",
)

# Launch the app
iface.launch(debug = True)

It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://4315c0ae785196202c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 752a7e60-c70b-4b3c-aceb-ed53147cbcde                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

🚀 Crew: crew
└── 📋 Task: 31ec3261-41f1-41ec-b300-b268265c0949
       Status: Executing Task...

🚀 Crew: crew
└── 📋 Task: 31ec3261-41f1-41ec-b300-b268265c0949
       Status: Executing Task...
    └── 🤖 Agent: Specjalista ds. Hoteli
            Status: In Progress

# Agent: Specjalista ds. Hoteli
## Task: Wyszukaj 5 hoteli w Lublin dla 2 osób dorosłych, z zameldowaniem 25-06-2025 i wymeldowaniem 29-06-2025.


...



# Agent: Analityk Jakości Hoteli
## Final Answer: 
I have found several hotel options in Lublin that fit within the budget of 500.0 PLN. Here are the details of the selected accommodations:

1. **Hotel Mercure Lublin Centrum**
   - **Location:** Al. Racławickie 12, 20-037 Lublin, Poland
   - **Rating:** 4.7/5 (555 reviews)
   - **Description:** 103 comfortable rooms, 7 apartments, modern conference rooms, restaurant WINESTONE, WiFi, gym, and parking. Close to the city center and cultural attractions.
   - **Price:** Approximately 400 PLN per night
   - **Website:** [Mercure Lublin Centrum](https://all.accor.com/hotel/3404/index.pl.shtml)

2. **EASY RENT Apartments - Aperta ART - Basen & SPA**
   - **Location:** Aleje Racławickie 10 125, piętro 6, 20-400 Lublin, Poland
   - **Rating:** 9.6/10 (114 reviews)
   - **Description:** Offers a one-bedroom apartment with a private pool, sauna, and fitness center. Free WiFi, air-conditioning, and a fully equipped kitchenette.
   - **Price:** A

🚀 Crew: crew
├── 📋 Task: 31ec3261-41f1-41ec-b300-b268265c0949
│      Assigned to: Specjalista ds. Hoteli
│      Status: ✅ Completed
│   └── 🤖 Agent: Specjalista ds. Hoteli
│           Status: ✅ Completed
└── 📋 Task: 1051c2a9-a7de-4221-ade7-62c43f4257ed
       Status: Executing Task...
    └── 🤖 Agent: Analityk Jakości Hoteli
            Status: ✅ Completed

🚀 Crew: crew
├── 📋 Task: 31ec3261-41f1-41ec-b300-b268265c0949
│      Assigned to: Specjalista ds. Hoteli
│      Status: ✅ Completed
│   └── 🤖 Agent: Specjalista ds. Hoteli
│           Status: ✅ Completed
└── 📋 Task: 1051c2a9-a7de-4221-ade7-62c43f4257ed
       Assigned to: Analityk Jakości Hoteli
       Status: ✅ Completed
    └── 🤖 Agent: Analityk Jakości Hoteli
            Status: ✅ Completed

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 1051c2a9-a7de-4221-ade7-62c43f4257ed                                                                     │
│  Agent: Analityk Jakości Hoteli                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 752a7e60-c70b-4b3c-aceb-ed53147cbcde                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Ten kod tworzy interfejs użytkownika dla funkcji `run_travel_agent_crew` za pomocą biblioteki Gradio i uruchamia aplikację.

1.  **Tworzenie interfejsu:** Linia `iface = gr.Interface(...)` tworzy obiekt interfejsu Gradio. Argumenty:
    *   `fn=run_travel_agent_crew`: Określa funkcję, która będzie wykonywana po przesłaniu danych przez użytkownika (w tym przypadku funkcja wyszukująca hotele).
    *   `inputs=[...]`: Definiuje pola wejściowe interfejsu:
        *   `gr.Textbox(label="Location")`: Pole tekstowe do wprowadzenia lokalizacji.
        *   `gr.Textbox(label="Check-in Date")`: Pole tekstowe do wprowadzenia daty zameldowania.
        *   `gr.Textbox(label="Check-out Date")`: Pole tekstowe do wprowadzenia daty wymeldowania.
        *   `gr.Number(label="Number of People")`: Pole numeryczne do wprowadzenia liczby osób.
        *   `gr.Number(label="Budget")`: Pole numeryczne do wprowadzenia budżetu.
    *   `outputs=gr.Markdown(label="Result")`: Definiuje pole wyjściowe interfejsu, które będzie wyświetlać wynik w formacie Markdown.
    *   `title="Travel Agent Crew"`: Ustawia tytuł aplikacji.
    *   `description="Enter your travel details to get hotel recommendations."`: Dodaje opis do aplikacji.
    *   `allow_flagging="never"`: Wyłącza możliwość oznaczania wyników przez użytkowników.

2.  **Uruchomienie aplikacji:** Linia `iface.launch(debug = True)` uruchamia aplikację Gradio w trybie debugowania. Tryb debugowania umożliwia łatwiejsze wykrywanie i rozwiązywanie problemów podczas działania aplikacji. Aplikacja będzie dostępna pod adresem URL wyświetlonym w konsoli.

W efekcie, ten kod tworzy prostą aplikację internetową z polami do wprowadzenia danych dotyczących podróży (lokalizacja, daty, liczba osób, budżet) i przyciskiem uruchomienia. Po przesłaniu danych aplikacja wywołuje funkcję `run_travel_agent_crew`, a wynik jest wyświetlany w formacie Markdown na stronie internetowej.

# Kroki w nieznane


*   Deployment: Spaces https://huggingface.co/spaces
*   Tutorial: https://www.gradio.app/guides/quickstart

